# 📼 Rebobina — narração com Qwen3-TTS (grátis, GPU do Colab)

**Antes de tudo:** menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4** → Salvar.

Como usar:
1. Rode as células **1** e **2** (instala e define a voz do canal).
2. **Primeira vez:** a célula 3 *cria* a voz do Rebobina e baixa `voz-rebobina.wav` + `voz-rebobina.txt`. **Guarde os dois em `canal/voz/`** do projeto: é a identidade sonora do canal.
   **Nas próximas vezes:** marque `USAR_VOZ_SALVA` e envie esses dois arquivos — assim todo episódio tem o mesmo narrador.
3. Célula 4: envie o `roteiro.json` do episódio.
4. Célula 5: gera uma faixa por cena e baixa `narracao-<slug>.zip`.
5. No Mac: coloque o zip em `episodios/<ep>/assets/voz/`, troque `"motor_voz"` para `"arquivos"` no roteiro e rode `python3 ferramentas/narrar.py episodios/<ep>`.

Modelo: Qwen3-TTS (Alibaba), licença Apache 2.0 — permite uso comercial.

In [ ]:
#@title 1 · Instalar (≈ 2 min)
!pip install -q -U qwen-tts soundfile
import torch, subprocess
assert torch.cuda.is_available(), "Sem GPU! Menu Ambiente de execução → Alterar tipo → GPU T4"
print("GPU:", torch.cuda.get_device_name(0))
# T4 não tem bfloat16 nem FlashAttention 2: usamos float16 + atenção padrão (sdpa)
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
ATTN = "sdpa"

In [ ]:
#@title 2 · Voz do canal
USAR_VOZ_SALVA = False  #@param {type:"boolean"}
#@markdown Descrição da voz (em inglês funciona melhor no modelo de design; ela só é usada na 1ª vez):
INSTRUCAO_VOZ = "Brazilian Portuguese male narrator in his 40s, deep warm baritone, neutral Brazilian accent, calm documentary pacing with subtle intrigue, very clear diction, natural pauses"  #@param {type:"string"}
TEXTO_REFERENCIA = "Toda fita tem um começo. Hoje, a gente rebobina até o momento em que essa história aconteceu de verdade, com datas, documentos e mapas."  #@param {type:"string"}
SEMENTE = 7  #@param {type:"integer"}

In [ ]:
#@title 3 · Criar (1ª vez) ou carregar a voz do Rebobina
import soundfile as sf, os
from google.colab import files
from qwen_tts import Qwen3TTSModel

if USAR_VOZ_SALVA:
    print("Envie voz-rebobina.wav e voz-rebobina.txt")
    up = files.upload()
    REF_WAV = "voz-rebobina.wav"
    TEXTO_REFERENCIA = open("voz-rebobina.txt", encoding="utf-8").read().strip()
else:
    torch.manual_seed(SEMENTE)
    design = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", device_map="cuda:0", dtype=DTYPE, attn_implementation=ATTN)
    wavs, sr = design.generate_voice_design(text=TEXTO_REFERENCIA, language="Portuguese", instruct=INSTRUCAO_VOZ)
    REF_WAV = "voz-rebobina.wav"
    sf.write(REF_WAV, wavs[0], sr)
    open("voz-rebobina.txt", "w", encoding="utf-8").write(TEXTO_REFERENCIA)
    del design; torch.cuda.empty_cache()
    from IPython.display import Audio, display
    display(Audio(REF_WAV))
    print("Gostou? Baixando voz-rebobina.wav e .txt — guarde em canal/voz/. Não gostou? Mude SEMENTE ou a descrição e rode de novo.")
    files.download("voz-rebobina.wav"); files.download("voz-rebobina.txt")

In [ ]:
#@title 4 · Enviar o roteiro.json do episódio
import json
up = files.upload()
nome = next(n for n in up if n.endswith(".json"))
roteiro = json.loads(up[nome].decode("utf-8"))
print(f"Episódio {roteiro['id']} · {roteiro['slug']} · {len(roteiro['cenas'])} cenas")

In [ ]:
#@title 5 · Gerar a narração (uma faixa por cena) e baixar o zip
import zipfile, numpy as np
clone = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-Base", device_map="cuda:0", dtype=DTYPE, attn_implementation=ATTN)
prompt = clone.create_voice_clone_prompt(ref_audio=REF_WAV, ref_text=TEXTO_REFERENCIA)

os.makedirs("cenas", exist_ok=True)
for c in roteiro["cenas"]:
    texto = c.get("fala_tts") or c["fala"]
    esperado = len(c["fala"].split()) / 2.7          # ~160 palavras/min
    for tentativa in range(4):                        # refaz se sair cortado ou "alucinado"
        torch.manual_seed(SEMENTE + tentativa)
        wavs, sr = clone.generate_voice_clone(text=texto, language="Portuguese", voice_clone_prompt=prompt)
        dur = len(wavs[0]) / sr
        if 0.5 * esperado <= dur <= 1.8 * esperado + 1.0:
            break
        print(f"  {c['id']}: {dur:.1f}s fora do esperado ({esperado:.1f}s) — tentando de novo")
    sf.write(f"cenas/{c['id']}.wav", wavs[0], sr)
    print(f"✓ {c['id']}  {dur:5.1f}s  {c['fala'][:60]}")

zip_nome = f"narracao-{roteiro['slug']}.zip"
with zipfile.ZipFile(zip_nome, "w") as z:
    for c in roteiro["cenas"]:
        z.write(f"cenas/{c['id']}.wav", f"{c['id']}.wav")
files.download(zip_nome)
print("Coloque o zip em episodios/<ep>/assets/voz/ e rode narrar.py com motor_voz = \"arquivos\".")